# BraTS 2021 exploratory data analysis
Run `scripts/prepare_dataset.py` first. All statistics below come from real metadata; no results are fabricated. For slice examples, **small** is the minimum-area positive slice, **medium** is closest to the median positive-slice area, and **large** is the maximum-area positive slice for the selected patient.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data.nifti import load_nifti
from src.data.preprocessing import convert_brats_mask_to_binary
from src.visualization.plotting import overlay_mask

In [ ]:
metadata = pd.read_csv(ROOT / 'data/metadata/dataset_metadata.csv')
display(metadata.head())
print('Patients:', len(metadata))
display(metadata[['height','width','depth','voxel_spacing_x','voxel_spacing_y','voxel_spacing_z']].describe())
display(metadata[['tumor_voxel_count','tumor_volume_mm3','positive_slice_count','negative_brain_slice_count']].describe())

In [ ]:
ordered = metadata.sort_values('tumor_volume_mm3').reset_index(drop=True)
example_rows = ordered.iloc[np.linspace(0, len(ordered)-1, min(3, len(ordered)), dtype=int)]
fig, axes = plt.subplots(len(example_rows), 3, figsize=(10, 4*len(example_rows)), squeeze=False)
for axes_row, (_, example) in zip(axes, example_rows.iterrows()):
    example_flair = load_nifti(example.flair_path).array
    example_mask = convert_brats_mask_to_binary(load_nifti(example.seg_path).array)
    z = int(np.argmax(example_mask.sum(axis=(0,1))))
    for axis, title, image, cmap in zip(axes_row, ['FLAIR','Ground truth','Overlay'], [example_flair[:,:,z], example_mask[:,:,z], overlay_mask(example_flair[:,:,z], example_mask[:,:,z])], ['gray','gray',None]):
        axis.imshow(image, cmap=cmap); axis.set_title(f'{example.patient_id} — {title}'); axis.axis('off')
plt.tight_layout()

In [ ]:
metadata['tumor_volume_mm3'].hist(bins=30); plt.title('Tumor volume distribution'); plt.xlabel('mm³'); plt.show()

In [ ]:
row = metadata.sort_values('tumor_voxel_count').iloc[len(metadata)//2]
flair = load_nifti(row.flair_path).array
mask = convert_brats_mask_to_binary(load_nifti(row.seg_path).array)
areas = mask.sum(axis=(0,1)); positive = np.flatnonzero(areas); negative = np.flatnonzero((flair != 0).any(axis=(0,1)) & (areas == 0))
positive_areas = areas[positive]; median_area = np.median(positive_areas)
indices = {'small': positive[np.argmin(positive_areas)], 'medium': positive[np.argmin(np.abs(positive_areas - median_area))], 'large': positive[np.argmax(positive_areas)], 'negative': negative[len(negative)//2]}
fig, axes = plt.subplots(len(indices), 3, figsize=(10, 12))
for axes_row, (label, z) in zip(axes, indices.items()):
    axes_row[0].imshow(flair[:,:,z], cmap='gray'); axes_row[0].set_title(f'{label}: FLAIR')
    axes_row[1].imshow(mask[:,:,z], cmap='gray'); axes_row[1].set_title('Ground truth')
    axes_row[2].imshow(overlay_mask(flair[:,:,z], mask[:,:,z])); axes_row[2].set_title('Overlay')
    [axis.axis('off') for axis in axes_row]
plt.tight_layout()